In [ ]:
import kagglehub
import os
import pandas as pd
from datetime import datetime

# Import Function

In [ ]:
def load_data(includeCancelations = True, keepna=True):
  path = kagglehub.dataset_download("mashlyn/online-retail-ii-uci")
  file_path = os.path.join(path, "online_retail_II.csv")
  df = pd.read_csv(file_path)

  df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
  df['Customer ID'] = pd.to_numeric(df['Customer ID'], downcast='integer')
  df["TotalPrice"] = df["Quantity"] * df["Price"]
  df = df[~df["StockCode"].str.startswith("TEST")]

  if not includeCancelations:
    df = df[~df["Invoice"].str.startswith("C")]

  if not keepna:
    df.dropna(subset=['Customer ID'], inplace=True)

  df['Customer ID'] = df['Customer ID'].astype('Int64')
  df.drop_duplicates(inplace=True)
  df.reset_index(drop=True, inplace=True)

  assert all(df["InvoiceDate"] > datetime(2009,1,1))
  assert all(df["InvoiceDate"] < datetime(2012,1,1))

  print("── Data loaded ──────────────────────────")
  print(f"Rows:              {len(df):,}")
  print(f"Date range:        {df['InvoiceDate'].min().date()} → {df['InvoiceDate'].max().date()}")
  print(f"Unique customers:  {df['Customer ID'].nunique():,}")
  print(f"Cancellations:     {df['Invoice'].str.startswith('C').sum():,}")
  print(f"Null Customer IDs: {df['Customer ID'].isna().sum():,}")
  print(f"─────────────────────────────────────────")

  return df

In [ ]:
def clean_data(df):
    """
    Cleaning decisions applied after load_data():

    Excluded:
    - Non-standard StockCodes: POST, DOT, M, D, S, C2, B,
      BANK CHARGES, AMAZONFEE, ADJUST — accounting/admin entries, not product sales
    - Zero price rows: warehouse adjustments, stock checks, samples (n=6,011)
    - Negative price rows (non-cancellation, prefix A): bad debt write-offs (n=5)
    - Null descriptions: uninformative, small volume (n=4,274)

    Retained with flag:
    - Extreme quantities (>10,000 units): possible wholesale orders,
      retained but noted as outliers (n=9)

    Null Customer IDs:
    - Retained in base cleaned data
    - Callers should filter on Customer ID as needed for customer-level analysis
    """

    NON_STANDARD_CODES = {
        'POST', 'DOT', 'M', 'D', 'S', 'C2', 'B',
        'BANK CHARGES', 'AMAZONFEE', 'ADJUST'
    }

    original_len = len(df)

    # Remove non-standard StockCodes
    df = df[~df['StockCode'].isin(NON_STANDARD_CODES)]

    # Remove zero and negative price rows (non-cancellation)
    df = df[~((df['Price'] <= 0) & ~df['Invoice'].str.startswith('C'))]

    # Remove null descriptions
    df = df[df['Description'].notna()]

    # Flag extreme quantities but retain
    df = df.copy()
    df['IsOutlier'] = df['Quantity'] > 10000

    print(f"── Cleaning summary ─────────────────────")
    print(f"Rows before: {original_len:,}")
    print(f"Rows after:  {len(df):,}")
    print(f"Removed:     {original_len - len(df):,}")
    print(f"Outlier rows flagged: {df['IsOutlier'].sum()}")
    print(f"─────────────────────────────────────────")

    return df

In [ ]:
df = load_data()


Using Colab cache for faster access to the 'online-retail-ii-uci' dataset.
── Data loaded ──────────────────────────
Rows:              1,033,019
Date range:        2009-12-01 → 2011-12-09
Unique customers:  5,940
Cancellations:     19,100
Null Customer IDs: 235,150
─────────────────────────────────────────


In [ ]:
df = clean_data(df)

── Cleaning summary ─────────────────────
Rows before: 1,033,019
Rows after:  1,021,374
Removed:     11,645
Outlier rows flagged: 7
─────────────────────────────────────────


# Question 1

In [ ]:
cancellations = df[df['Invoice'].str.startswith('C')]
sales = df[~df['Invoice'].str.startswith('C')]

gross_revenue = sales['TotalPrice'].sum()
cancellation_revenue = cancellations['TotalPrice'].sum()
net_revenue = gross_revenue + cancellation_revenue

print(f"Gross Revenue:    £{gross_revenue:>14,.2f}")
print(f"Cancellation value:  £{cancellation_revenue:>14,.2f}")
print(f"Net revenue:         £{net_revenue:>14,.2f}")
print(f"Cancellation rate:   {abs(cancellation_revenue) / gross_revenue * 100:.2f}%")

Gross Revenue:    £ 19,646,363.91
Cancellation value:  £   -724,465.56
Net revenue:         £ 18,921,898.35
Cancellation rate:   3.69%


# Question 2

In [ ]:
cancellations

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,TotalPrice,IsOutlier
178,C489449,22087,PAPER BUNTING WHITE LACE,-12,2009-12-01 10:33:00,2.95,16321,Australia,-35.40,False
179,C489449,85206A,CREAM FELT EASTER EGG BASKET,-6,2009-12-01 10:33:00,1.65,16321,Australia,-9.90,False
180,C489449,21895,POTTING SHED SOW 'N' GROW SET,-4,2009-12-01 10:33:00,4.25,16321,Australia,-17.00,False
181,C489449,21896,POTTING SHED TWINE,-6,2009-12-01 10:33:00,2.10,16321,Australia,-12.60,False
182,C489449,22083,PAPER CHAIN KIT RETRO SPOT,-12,2009-12-01 10:33:00,2.95,16321,Australia,-35.40,False
...,...,...,...,...,...,...,...,...,...,...
1031564,C581490,22178,VICTORIAN GLASS HANGING T-LIGHT,-12,2011-12-09 09:57:00,1.95,14397,United Kingdom,-23.40,False
1031565,C581490,23144,ZINC T-LIGHT HOLDER STARS SMALL,-11,2011-12-09 09:57:00,0.83,14397,United Kingdom,-9.13,False
1032824,C581568,21258,VICTORIAN SEWING BOX LARGE,-5,2011-12-09 11:57:00,10.95,15311,United Kingdom,-54.75,False
1032825,C581569,84978,HANGING HEART JAR T-LIGHT HOLDER,-1,2011-12-09 11:58:00,1.25,17315,United Kingdom,-1.25,False


In [ ]:
cancellation_by_month = (cancellations.groupby(cancellations['InvoiceDate'].dt.strftime('%Y-%m'))['TotalPrice']
                  .sum()
                  .reset_index()
                  .rename(columns={'InvoiceDate': 'Month', 'TotalPrice': 'Cancellations'}))

sales_by_month = (sales.groupby(sales['InvoiceDate'].dt.strftime('%Y-%m'))['TotalPrice']
                  .sum()
                  .reset_index()
                  .rename(columns={'InvoiceDate': 'Month', 'TotalPrice': 'Gross Revenue'}))

totals_by_month = sales_by_month.merge(cancellation_by_month, left_on='Month', right_on='Month')
totals_by_month['Net Revenue'] = totals_by_month['Gross Revenue'] + totals_by_month['Cancellations']
totals_by_month['Cancellation Rate'] = (abs(totals_by_month['Cancellations']) / totals_by_month['Gross Revenue'] * 100).round(2)
totals_by_month

,Month,Gross Revenue,Cancellations,Net Revenue,Cancellation Rate
0,2009-12,798118.530,-19558.08,778560.450,2.45
1,2010-01,612365.502,-7535.19,604830.312,1.23
2,2010-02,537932.646,-12575.44,525357.206,2.34
3,2010-03,761748.531,-9603.85,752144.681,1.26
4,2010-04,646455.562,-10025.26,636430.302,1.55
5,2010-05,643585.640,-37192.89,606392.750,5.78
6,2010-06,697384.870,-26373.46,671011.410,3.78
7,2010-07,633079.420,-17132.73,615946.690,2.71
8,2010-08,674192.890,-12397.60,661795.290,1.84
9,2010-09,869277.161,-27686.42,841590.741,3.18
